In [1]:
import os
os.environ["OLLAMA_MODELS"] = "D:\\ollama_models"

In [3]:
import warnings
warnings.filterwarnings('ignore')

### Step 1: Document Loading & Splitting

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
loader = PyPDFLoader("apple-privacy-policy-en-ww.pdf")
documents = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Split document into {len(chunks)} chunks.")

Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)


Split document into 37 chunks.


### Step 2: Creating the Vector Database

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Vector DB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vector database created successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector database created successfully!


### Step 3: Initialize the LLM (Ollama)

In [10]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="mistral")

### Step 4: Creating the RAG Pipeline

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Prompt
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question ONLY using the provided context.

<context>
{context}
</context>

Question: {question}
""")

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LangChain Expression Language (LCEL) Chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

### Step 5: Ask the Question

In [19]:
question = "According to the document, why user's personal data is used by Apple?"

response = rag_chain.invoke(question)

print("\n--- Answer ---")
print(response)

ResponseError: model requires more system memory (4.6 GiB) than is available (1.8 GiB) (status code: 500)